# Existence and direction priors

AUPRC is recalculated from the stored prior matrices and true graphs for the five search datasets in each family. Model fitting and prior estimation are not repeated.


In [1]:
from pathlib import Path
import sys, os, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/run_model.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "experiments")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import final_visualization as fv
import result_tables as th
fv.apply_style()


In [2]:
from metrics import existence_auprc, directed_auprc

config = json.loads((ROOT / "configs/prism_study.json").read_text())
rows = []
with np.load(ROOT / "results/prism/priors/search_estimates.npz", allow_pickle=False) as estimates:
    for family in config["families"]:
        for seed in config["search_dataset_seeds"]:
            with th.load_sample("model_selection", family, seed) as observations:
                truth = observations["true_adj"]
            prefix = f"{family}__seed_{seed}__"
            values = {name: estimates[prefix + name] for name in
                      ("r_G", "r_H_raw", "r_H_z", "d_binary", "d_binary_z", "d_bootstrap", "d_bootstrap_z")}
            def record(score, representation, matrix, target, alpha=None):
                metric = existence_auprc if target == "skeleton" else directed_auprc
                rows.append(dict(dataset=family, dataset_seed=seed, score=score,
                                 representation=representation, alpha=alpha, target=target,
                                 auprc=metric(truth, matrix)))
            record("GLasso", "correlation", values["r_G"], "skeleton")
            for representation, key in (("raw", "r_H_raw"), ("standardized", "r_H_z")):
                record("Hessian", representation, values[key], "skeleton")
                for alpha in (.25, .5, .75):
                    combined = alpha * values["r_G"] + (1 - alpha) * values[key]
                    record("Combined", representation, combined, "skeleton", alpha)
            for label, raw, standardized in (("Binary", "d_binary", "d_binary_z"),
                                             ("Subsampling", "d_bootstrap", "d_bootstrap_z")):
                record(label, "raw", values["r_G"] * values[raw], "directed")
                record(label, "standardized", values["r_G"] * values[standardized], "directed")
scores = pd.DataFrame(rows)


## Existence priors

Average precision over unordered variable pairs against the true graph skeleton. Values are mean ± sample standard deviation across dataset seeds.


In [3]:
prior_table = th.prior_quality(scores)
display(prior_table.iloc[:5].reset_index(drop=True))


,Prior,"ER, non-Gaussian mixed",MultiOmics,Oncogene,Macro
0,GLasso,0.625 ± 0.022,0.772 ± 0.078,0.659 ± 0.081,0.685 ± 0.044
1,"Hessian, raw",0.473 ± 0.031,0.602 ± 0.032,0.703 ± 0.057,0.593 ± 0.031
2,"Hessian, standardized",0.479 ± 0.038,0.577 ± 0.032,0.675 ± 0.067,0.577 ± 0.031
3,"Combined, raw",0.573 ± 0.022,0.747 ± 0.042,0.746 ± 0.096,0.689 ± 0.037
4,"Combined, standardized",0.573 ± 0.022,0.737 ± 0.054,0.715 ± 0.103,0.675 ± 0.042


## Direction priors

Average precision over ordered variable pairs, using GLasso ranks multiplied by the direction score.


In [4]:
display(prior_table.iloc[5:].reset_index(drop=True))


,Prior,"ER, non-Gaussian mixed",MultiOmics,Oncogene,Macro
0,"Binary, raw",0.482 ± 0.081,0.396 ± 0.113,0.367 ± 0.109,0.415 ± 0.028
1,"Binary, standardized",0.265 ± 0.052,0.348 ± 0.104,0.403 ± 0.107,0.338 ± 0.036
2,"Subsampling, raw",0.533 ± 0.057,0.405 ± 0.104,0.382 ± 0.141,0.440 ± 0.030
3,"Subsampling, standardized",0.243 ± 0.043,0.338 ± 0.095,0.418 ± 0.111,0.333 ± 0.045


## Mixture sensitivity

Additional mixtures of the same GLasso and Hessian estimates.


In [5]:
mixtures = th.prior_quality(scores, include_extra=True)
display(mixtures.loc[mixtures.Prior.str.contains("alpha=")].reset_index(drop=True))


,Prior,"ER, non-Gaussian mixed",MultiOmics,Oncogene,Macro
0,"Combined, raw, alpha=0.25",0.522 ± 0.023,0.676 ± 0.045,0.737 ± 0.077,0.645 ± 0.032
1,"Combined, raw, alpha=0.75",0.611 ± 0.023,0.778 ± 0.045,0.725 ± 0.094,0.705 ± 0.035
2,"Combined, standardized, alpha=0.25",0.528 ± 0.035,0.669 ± 0.048,0.709 ± 0.089,0.636 ± 0.038
3,"Combined, standardized, alpha=0.75",0.607 ± 0.020,0.776 ± 0.055,0.708 ± 0.101,0.697 ± 0.042


## Raw versus standardized direction estimates

Paired comparison of the subsampling direction score on the same datasets. The 95% interval uses 5,000 paired bootstrap draws within each family, with equal weight across families.


In [6]:
direction = (scores.loc[scores.score.eq("Subsampling")]
    .pivot(index=["dataset", "dataset_seed"], columns="representation", values="auprc"))
direction_summary = direction.groupby("dataset")[["raw", "standardized"]].mean()
direction_summary.loc["Macro"] = direction.mean()
display(direction_summary.rename(index=th.FAMILY_LABELS))
differences = [(direction.loc[family, "raw"] - direction.loc[family, "standardized"]).to_numpy()
               for family in config["families"]]
rng = np.random.default_rng(20260919)
draws = np.array([np.mean([rng.choice(d, len(d), replace=True).mean() for d in differences])
                  for _ in range(5000)])
lower, upper = np.quantile(draws, [.025, .975])
display(pd.DataFrame([dict(raw_minus_standardized=float(np.mean(differences)),
                           ci95_lower=lower, ci95_upper=upper)]))


representation,raw,standardized
dataset,,
"ER, non Gaussian mixed",0.533486,0.243152
"MultiOmics, Laplace mixed",0.404818,0.338230
"Oncogene, Laplace mixed",0.381537,0.418387
Macro,0.439947,0.333257


,raw_minus_standardized,ci95_lower,ci95_upper
0,0.10669,0.047059,0.160322
